<a href="https://colab.research.google.com/github/thaoly-corner/reddit-iphone-feedback-pipeline/blob/dev/notebooks/historical_posts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Preparation steps**

 **Mount Google Drive**

In [ ]:
from google.colab import drive

# Kết nối (Mount) với Google Drive cá nhân để lưu trữ dữ liệu lâu dài
drive.mount("/content/drive")

Mounted at /content/drive


**Install depencencies**

In [ ]:
# Cập nhật danh sách gói và cài đặt aria2 để tải torrent tốc độ cao
!apt-get update -qq
!apt-get install -y aria2

# Cài đặt các thư viện Python cần thiết để xử lý nén zstd, phân tích dữ liệu và định dạng Parquet
!pip install -q zstandard pandas pyarrow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libcares2
The following NEW packages will be installed:
  aria2 libaria2-0 libcares2
0 upgraded, 3 newly installed, 0 to remove and 59 not upgraded.
Need to get 1,566 kB of archives.
After this operation, 5,692 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 libcares2 amd64 1.27.0-1.0ubuntu1 [73.7 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/universe amd64 libaria2-0 amd64 1.37.0+debian-1build3 [1,105 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble/universe amd64 aria2 amd64 1.37.0+debian-1build3 [387 kB]
Fetched 1,566 kB in 1s (1,660 kB/s)
Selecting previously unselected package libcar

**Import necessary libraries**

In [ ]:
# Import các thư viện hệ thống và tiện ích xử lý tệp tin
from pathlib import Path
import json
import math
import re
import csv
from collections import Counter, defaultdict
from datetime import datetime, timezone

# Import các thư viện phân tích dữ liệu, nén và thanh tiến trình
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import zstandard as zstd
from tqdm.auto import tqdm

### **PROJECT CONFIGURATION**

In [ ]:
# Định nghĩa các đường dẫn thư mục gốc và thư mục lưu trữ sản phẩm trung gian trên Google Drive
PROJECT_ROOT = Path("/content/drive/MyDrive/DE")
RAW_LOCAL_DIR = Path("/content/reddit_2026_07")
FILTERED_DIR = PROJECT_ROOT / "filtered"
PARQUET_DIR = PROJECT_ROOT / "parquet"
PROFILE_DIR = PROJECT_ROOT / "profiling"

# Tự động tạo cấu trúc thư mục nếu chưa tồn tại
for folder in [
    RAW_LOCAL_DIR,
    FILTERED_DIR,
    PARQUET_DIR,
    PROFILE_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /content/drive/MyDrive/DE


### **Upload file `.torrent`**

**Upload `.torrent` file**

In [ ]:
from google.colab import files

# Hiển thị giao diện tải tệp tin từ máy cục bộ lên môi trường Google Colab
uploaded = files.upload()

Saving reddit-e04a4fda12826ab1d181eef6512b36aca63c70ff.torrent to reddit-e04a4fda12826ab1d181eef6512b36aca63c70ff.torrent


**Xác định torrent file**

In [ ]:
# Trích xuất danh sách tên các tệp tin torrent vừa được tải lên thành công
uploaded_files = list(uploaded.keys())

print(uploaded_files)

['reddit-e04a4fda12826ab1d181eef6512b36aca63c70ff.torrent']


**Determine path to `.torrent` file**

In [ ]:
# Xác định chính xác đường dẫn tuyệt đối của tệp tin .torrent vừa tải lên
TORRENT_FILE = Path("/content") / uploaded_files[0]

print("Torrent file:")
print(TORRENT_FILE)

Torrent file:
/content/reddit-e04a4fda12826ab1d181eef6512b36aca63c70ff.torrent


**List of files within torrent**

In [ ]:
# Liệt kê thông tin và danh sách các tệp tin chứa trong torrent để tìm chỉ số (idx) cần tải
!aria2c -S "$TORRENT_FILE"

>>> Printing the contents of file '/content/reddit-e04a4fda12826ab1d181eef6512b36aca63c70ff.torrent'...
*** BitTorrent File Information ***
Comment: Reddit comments and submissions from 2026-07
Creation Date: Sun, 09 Aug 2026 19:54:09 GMT
Created By: torrenttools/0.6.2
Mode: multi
Announce:
 https://academictorrents.com/announce.php
 udp://tracker.opentrackr.org:1337/announce
Info Hash: e04a4fda12826ab1d181eef6512b36aca63c70ff
Piece Length: 16MiB
The Number of Pieces: 4785
Total Length: 74GiB (80,272,561,702)
URL List:
 https://seed.pullpush.io/file/RaiderB/
Name: reddit
Magnet URI: magnet:?xt=urn:btih:E04A4FDA12826AB1D181EEF6512B36ACA63C70FF&dn=reddit&tr=https%3A%2F%2Facademictorrents.com%2Fannounce.php&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce
Files:
idx|path/length
===+===========================================================================
  1|./reddit/comments/RC_2026-07.zst
   |53GiB (57,061,907,266)
---+-----------------------------------------------------------

###**Dowload submissions**

In [ ]:
# Tiến hành tải tệp tin cụ thể từ torrent (ở đây chọn file số 2 là submissions) thông qua công cụ aria2c
!aria2c \
    --select-file=2 \
    --seed-time=0 \
    --file-allocation=none \
    "$TORRENT_FILE" \
    -d /content/reddit_2026_07


10/18 07:43:27 [NOTICE] Downloading 1 item(s)

10/18 07:43:27 [ERROR] Exception caught while loading DHT routing table from /root/.cache/aria2/dht.dat
Exception: [DHTRoutingTableDeserializer.cc:82] errorCode=1 Failed to load DHT routing table from /root/.cache/aria2/dht.dat

10/18 07:43:27 [NOTICE] IPv4 DHT: listening on UDP port 6958

10/18 07:43:27 [NOTICE] IPv4 BitTorrent: listening on TCP port 6901

10/18 07:43:27 [ERROR] IPv6 BitTorrent: failed to bind TCP port 6901
Exception: [SocketCore.cc:312] errorCode=1 Failed to bind a socket, cause: Name or service not known

10/18 07:43:27 [ERROR] CUID#19 - Download aborted. URI=https://seed.pullpush.io/file/RaiderB/reddit/comments/RC_2026-07.zst
Exception: [AbstractCommand.cc:351] errorCode=8 URI=https://seed.pullpush.io/file/RaiderB/reddit/comments/RC_2026-07.zst
  -> [HttpResponse.cc:81] errorCode=8 Invalid range header. Request: 57059311616-57061907265/57061907266, Response: 0-57061907265/57061907266

10/18 07:43:28 [ERROR] CUID#20 - 

**Examine dowloaded file**

In [ ]:
# Quét toàn bộ thư mục tải xuống để kiểm tra sự tồn tại và kích thước các file đã hoàn thành tải
for file in RAW_LOCAL_DIR.rglob("*"):
    if file.is_file():
        size_gb = file.stat().st_size / (1024 ** 3)
        print(
            file,
            f"{size_gb:.2f} GB"
        )

/content/reddit_2026_07/reddit.aria2 0.00 GB
/content/reddit_2026_07/reddit/submissions/RS_2026-07.zst 21.62 GB
/content/reddit_2026_07/reddit/comments/RC_2026-07.zst 53.14 GB


**Tìm submissions tự động**

In [ ]:
# Tự động định vị tệp dữ liệu nén submissions (RS_2026-07.zst) trong thư mục vừa tải về
submission_files = list(
    RAW_LOCAL_DIR.rglob("RS_2026-07.zst")
)

print(submission_files)

[PosixPath('/content/reddit_2026_07/reddit/submissions/RS_2026-07.zst')]


In [ ]:
# Kiểm tra lỗi nếu không tìm thấy tệp, sau đó gán đường dẫn vào biến POSTS_RAW để xử lý tiếp
if not submission_files:
    raise FileNotFoundError(
        "Không tìm thấy RS_2026-07.zst"
    )

POSTS_RAW = submission_files[0]

print("Raw submissions:")
print(POSTS_RAW)

Raw submissions:
/content/reddit_2026_07/reddit/submissions/RS_2026-07.zst


**Đọc `.zst`**

In [ ]:
# Thiết lập các đường dẫn đầu ra cho dữ liệu JSONL đã lọc và dữ liệu chuyển đổi sang định dạng Parquet
POSTS_FILTERED = (
    FILTERED_DIR /
    "iphone_posts_2026_07.jsonl"
)

POSTS_PARQUET = (
    PARQUET_DIR /
    "iphone_posts_2026_07.parquet"
)

print(POSTS_FILTERED)
print(POSTS_PARQUET)

/content/drive/MyDrive/DE/filtered/iphone_posts_2026_07.jsonl
/content/drive/MyDrive/DE/parquet/iphone_posts_2026_07.parquet


In [ ]:
def stream_zst_json(
    path,
    chunk_size=1024 * 1024
):
    """
    Đọc file .zst chứa NDJSON/JSONL theo streaming bằng thư viện zstandard.

    Hàm này yield từng JSON object một mà không cần giải nén toàn bộ tệp tin ra ổ đĩa,
    giúp tối ưu hóa và tiết kiệm bộ nhớ RAM cũng như dung lượng đĩa ảo.
    """
    dctx = zstd.ZstdDecompressor(
        max_window_size=2**31
    )

    with open(path, "rb") as compressed_file:
        with dctx.stream_reader(
            compressed_file
        ) as reader:
            buffer = ""
            while True:
                chunk = reader.read(
                    chunk_size
                )
                if not chunk:
                    break
                buffer += chunk.decode(
                    "utf-8",
                    errors="replace"
                )
                lines = buffer.split("\n")
                # Dòng cuối có thể chưa hoàn chỉnh nên giữ lại trong buffer
                buffer = lines.pop()
                for line in lines:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        yield json.loads(line)
                    except json.JSONDecodeError:
                        continue

            # Xử lý phần dữ liệu còn lại cuối cùng trong buffer
            if buffer.strip():
                try:
                    yield json.loads(
                        buffer
                    )
                except json.JSONDecodeError:
                    pass

**Đọc thử 3 records**

In [ ]:
# Đọc thử 3 bản ghi đầu tiên của file nén gốc để hiểu cấu trúc dữ liệu JSON thô
for i, record in enumerate(
    stream_zst_json(POSTS_RAW)
):
    print(
        json.dumps(
            record,
            ensure_ascii=False,
            indent=2
        )
    )
    print("=" * 100)
    if i >= 2:
        break

**Lọc `r/iphone`**

In [ ]:
def extract_subreddit(
    input_path,
    output_path,
    subreddit="iphone",
    progress_every=1_000_000
):
    """
    Đọc trực tiếp stream của dữ liệu Reddit gốc và lọc ghi lại các bản ghi
    thuộc về một subreddit cụ thể (mặc định là 'iphone') vào file đích.
    """
    total = 0
    matched = 0
    missing_subreddit = 0

    output_path = Path(output_path)

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as out:
        for record in stream_zst_json(
            input_path
        ):
            total += 1
            sub = record.get(
                "subreddit"
            )
            if sub is None:
                missing_subreddit += 1
                continue

            if (
                str(sub).strip().lower()
                == subreddit.lower()
            ):
                out.write(
                    json.dumps(
                        record,
                        ensure_ascii=False
                    )
                    + "\n"
                )
                matched += 1

            # In nhật ký tiến độ xử lý định kỳ
            if (
                total %
                progress_every == 0
            ):
                print(
                    f"Processed: "
                    f"{total:,} | "
                    f"Matched r/{subreddit}: "
                    f"{matched:,}"
                )

    print("\n===== EXTRACTION COMPLETED =====")
    print(f"Total records: {total:,}")
    print(f"Matched r/{subreddit}: {matched:,}")
    print(f"Missing subreddit: {missing_subreddit:,}")

    if total > 0:
        print(
            "Match rate:",
            round(
                matched / total * 100,
                6
            ),
            "%"
        )

    return {
        "total_records": total,
        "matched_records": matched,
        "missing_subreddit": missing_subreddit
    }

In [ ]:
# Thực hiện chạy hàm lọc dữ liệu thô cho r/iphone và ghi nhận kết quả tổng hợp
posts_extract_summary = (
    extract_subreddit(
        input_path=POSTS_RAW,
        output_path=POSTS_FILTERED,
        subreddit="iphone"
    )
)

Processed: 1,000,000 | Matched r/iphone: 91
Processed: 2,000,000 | Matched r/iphone: 180
Processed: 3,000,000 | Matched r/iphone: 282
Processed: 4,000,000 | Matched r/iphone: 370
Processed: 5,000,000 | Matched r/iphone: 477
Processed: 6,000,000 | Matched r/iphone: 561
Processed: 7,000,000 | Matched r/iphone: 672
Processed: 8,000,000 | Matched r/iphone: 758
Processed: 9,000,000 | Matched r/iphone: 843
Processed: 10,000,000 | Matched r/iphone: 937
Processed: 11,000,000 | Matched r/iphone: 1,021
Processed: 12,000,000 | Matched r/iphone: 1,114
Processed: 13,000,000 | Matched r/iphone: 1,216
Processed: 14,000,000 | Matched r/iphone: 1,296
Processed: 15,000,000 | Matched r/iphone: 1,377
Processed: 16,000,000 | Matched r/iphone: 1,479
Processed: 17,000,000 | Matched r/iphone: 1,563
Processed: 18,000,000 | Matched r/iphone: 1,650
Processed: 19,000,000 | Matched r/iphone: 1,728
Processed: 20,000,000 | Matched r/iphone: 1,843
Processed: 21,000,000 | Matched r/iphone: 1,910
Processed: 22,000,000 

**Kích thước dữ liệu sau khi lọc**

In [ ]:
# Kiểm tra dung lượng tệp tin JSONL mới sau khi đã được lọc thành công
size_mb = (
    POSTS_FILTERED.stat().st_size
    / (1024 ** 2)
)

print(
    f"Filtered file size: "
    f"{size_mb:.2f} MB"
)

Filtered file size: 18.00 MB


**Preview 3 posts**

In [ ]:
def preview_jsonl(
    path,
    n=3
):
    """
    Hàm tiện ích giúp đọc nhanh n bản ghi đầu tiên của một tệp tin JSONL.
    """
    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            record = json.loads(line)
            print(
                json.dumps(
                    record,
                    indent=2,
                    ensure_ascii=False
                )
            )
            print(
                "-" * 100
            )

In [ ]:
# Hiển thị chi tiết cấu trúc 3 bản ghi đầu tiên trong tệp đã lọc r/iphone
preview_jsonl(
    POSTS_FILTERED,
    n=3
)

{
  "_meta": {
    "edited_title": "[ Removed by moderator ]",
    "removal_type": "moderator",
    "retrieved_2nd_on": 1782993660,
    "was_deleted_later": true
  },
  "all_awardings": [],
  "allow_live_comments": false,
  "approved_at_utc": null,
  "approved_by": null,
  "archived": false,
  "author": "Joseph930",
  "author_flair_background_color": null,
  "author_flair_css_class": null,
  "author_flair_richtext": [],
  "author_flair_template_id": null,
  "author_flair_text": null,
  "author_flair_text_color": null,
  "author_flair_type": "text",
  "author_fullname": "t2_5aru5gfh",
  "author_is_blocked": false,
  "author_patreon_flair": false,
  "author_premium": false,
  "awarders": [],
  "banned_at_utc": null,
  "banned_by": null,
  "can_gild": false,
  "can_mod_post": false,
  "category": null,
  "clicked": false,
  "content_categories": null,
  "contest_mode": false,
  "created": 1782864042,
  "created_utc": 1782864042,
  "crosspost_parent": "t3_1uk5weu",
  "crosspost_parent_list

In [ ]:
# In thu gọn các thuộc tính quan trọng của 5 bản ghi đầu tiên để xác thực sơ bộ
with open(
    POSTS_FILTERED,
    "r",
    encoding="utf-8"
) as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        obj = json.loads(line)
        print({
            "id": obj.get("id"),
            "subreddit": obj.get("subreddit"),
            "author": obj.get("author"),
            "title": obj.get("title"),
            "created_utc": obj.get("created_utc"),
            "score": obj.get("score"),
            "num_comments": obj.get("num_comments")
        })

{'id': '1uk5x5b', 'subreddit': 'iphone', 'author': 'Joseph930', 'title': 'iphone 14 pro max + ios 18.6.2 and Glad', 'created_utc': 1782864042, 'score': 1, 'num_comments': 1}
{'id': '1uk61nf', 'subreddit': 'iphone', 'author': 'Exotic-Ad-7209', 'title': 'Too many typos', 'created_utc': 1782864351, 'score': 5, 'num_comments': 14}
{'id': '1uk63iu', 'subreddit': 'iphone', 'author': 'AlmxstAlive', 'title': 'Why', 'created_utc': 1782864492, 'score': 16, 'num_comments': 11}
{'id': '1uk6nwa', 'subreddit': 'iphone', 'author': '_straightasmyhair', 'title': 'Would an iPhone 3 still work in 2026?', 'created_utc': 1782866038, 'score': 0, 'num_comments': 36}
{'id': '1uk6o3z', 'subreddit': 'iphone', 'author': 'frenetic_alien', 'title': 'Is there a way to tap to change focus point, but not change exposure along with it, in the camera app?', 'created_utc': 1782866054, 'score': 2, 'num_comments': 2}


# **Profiling**

In [ ]:
# Tải tệp dữ liệu r/iphone vào cấu trúc Pandas DataFrame để tiến hành phân tích profiling
df_posts = pd.read_json(
    POSTS_FILTERED,
    lines=True
)

print("Rows:", len(df_posts))
print("Columns:", len(df_posts.columns))

Rows: 4205
Columns: 121


In [ ]:
# Xem tóm tắt tổng thể về kiểu dữ liệu và số lượng non-null của các cột trong dataframe
df_posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4205 entries, 0 to 4204
Columns: 121 entries, _meta to author_cakeday
dtypes: bool(28), float64(28), int64(14), object(51)
memory usage: 3.1+ MB


In [ ]:
# Thống kê tổng số lượng bản ghi, số cột, số bài đăng duy nhất và số tác giả độc nhất
overview = {
    "row_count": len(df_posts),
    "column_count": len(df_posts.columns),
    "unique_posts": (
        df_posts["id"].nunique()
        if "id" in df_posts.columns
        else None
    ),
    "duplicate_ids": (
        df_posts["id"].duplicated().sum()
        if "id" in df_posts.columns
        else None
    ),
    "unique_authors": (
        df_posts["author"].nunique()
        if "author" in df_posts.columns
        else None
    )
}

overview

{'row_count': 4205,
 'column_count': 121,
 'unique_posts': 4205,
 'duplicate_ids': np.int64(0),
 'unique_authors': 3724}

In [ ]:
# Phân tích và thống kê số lượng cũng như tỷ lệ phần trăm các giá trị bị thiếu (NaN) trên mỗi trường dữ liệu
missing = pd.DataFrame({
    "column": df_posts.columns,
    "missing_count": [
        df_posts[c].isna().sum()
        for c in df_posts.columns
    ],
    "missing_percent": [
        round(
            df_posts[c].isna().mean() * 100,
            2
        )
        for c in df_posts.columns
    ]
})

missing = missing.sort_values(
    "missing_percent",
    ascending=False
)

missing

,column,missing_count,missing_percent
4,approved_by,4205,100.0
3,approved_at_utc,4205,100.0
20,banned_by,4205,100.0
23,category,4205,100.0
19,banned_at_utc,4205,100.0
...,...,...,...
108,url,0,0.0
106,ups,0,0.0
105,treatment_tags,0,0.0
107,upvote_ratio,0,0.0


In [ ]:
# Đếm số lượng bài viết bị xóa ([deleted]), bị gỡ ([removed]) hoặc không có nội dung trong trường selftext
if "selftext" in df_posts.columns:
    status_counts = (
        df_posts["selftext"]
        .fillna("")
        .astype(str)
        .value_counts()
    )
    deleted_count = (
        df_posts["selftext"]
        .fillna("")
        .eq("[deleted]")
        .sum()
    )
    removed_count = (
        df_posts["selftext"]
        .fillna("")
        .eq("[removed]")
        .sum()
    )
    empty_count = (
        df_posts["selftext"]
        .fillna("")
        .str.strip()
        .eq("")
        .sum()
    )

    print("Deleted:", deleted_count)
    print("Removed:", removed_count)
    print("Empty:", empty_count)

Deleted: 17
Removed: 839
Empty: 674


##**Timestamp**

In [ ]:
# Chuyển đổi mốc thời gian Unix timestamp sang đối tượng datetime định dạng chuẩn UTC
df_posts["created_datetime"] = pd.to_datetime(
    df_posts["created_utc"],
    unit="s",
    utc=True,
    errors="coerce"
)

In [ ]:
# Xem khung thời gian bắt đầu và kết thúc của các bài đăng được thu thập trong tập dữ liệu
print(
    "From:",
    df_posts["created_datetime"].min()
)
print(
    "To:",
    df_posts["created_datetime"].max()
)
print(
    "Invalid:",
    df_posts["created_datetime"].isna().sum()
)

From: 2026-07-01 00:00:42+00:00
To: 2026-07-31 23:58:46+00:00
Invalid: 0


**Kiểm tra subreddit**

In [ ]:
# Thống kê tần suất để đảm bảo 100% dữ liệu đã lọc thuộc về r/iphone
df_posts["subreddit"].value_counts()

,count
subreddit,
iphone,4205


**Numeric profile**

In [ ]:
# Mô tả tổng quan phân phối số liệu thống kê như điểm số, số bình luận và tỷ lệ upvote
numeric_columns = [
    column
    for column in [
        "score",
        "num_comments",
        "upvote_ratio"
    ]
    if column in df_posts.columns
]

numeric_profile = (
    df_posts[numeric_columns]
    .describe()
    .T
)

numeric_profile

,count,mean,std,min,25%,50%,75%,max
score,4205.0,14.969084,109.186142,0.00,0.0,1.00,1.0,2894.0
num_comments,4205.0,8.665874,39.630241,0.00,1.0,2.00,6.0,1456.0
upvote_ratio,4205.0,0.675234,0.303850,0.04,0.4,0.67,1.0,1.0


**Posts per day**

In [ ]:
# Gom nhóm và thống kê số lượng bài đăng được tạo theo từng ngày trong tháng
df_posts["date"] = (
    df_posts["created_datetime"]
    .dt.date
)

posts_per_day = (
    df_posts
    .groupby("date")
    .size()
    .reset_index(
        name="post_count"
    )
)

posts_per_day

,date,post_count
0,2026-07-01,125
1,2026-07-02,154
2,2026-07-03,135
3,2026-07-04,128
4,2026-07-05,144
5,2026-07-06,126
6,2026-07-07,144
7,2026-07-08,134
8,2026-07-09,149
9,2026-07-10,126


In [ ]:
# Tính toán độ dài (ký tự) của tiêu đề và nội dung tự sự của bài đăng
df_posts["title_length"] = (
    df_posts["title"]
    .fillna("")
    .astype(str)
    .str.len()
)

df_posts["selftext_length"] = (
    df_posts["selftext"]
    .fillna("")
    .astype(str)
    .str.len()
)

In [ ]:
# Thống kê mô tả về độ dài chuỗi ký tự tiêu đề bài đăng và nội dung chi tiết
df_posts[
    [
        "title_length",
        "selftext_length"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
title_length,4205.0,46.304637,33.492550,1.0,24.0,38.0,59.0,300.0
selftext_length,4205.0,292.872533,518.129824,0.0,9.0,144.0,394.0,16006.0


**Check sample thật**

In [ ]:
# Lấy ngẫu nhiên mẫu gồm 10 bài đăng để đánh giá thực tế chất lượng nội dung
sample_cols = [
    "title",
    "selftext",
    "score",
    "num_comments"
]

df_posts[sample_cols].sample(
    min(10, len(df_posts)),
    random_state=42
)

,title,selftext,score,num_comments
676,How to turn off automatic translation from dow...,"Hello, some (not all) apps get automatically t...",0,1
1831,iPhone 17 Pro Max bending,So I bought the newest flagship in January \...,38,86
3733,I wish the text messages could be archived and...,Apple has added a few organizational features ...,2,0
3734,Is there a way to make screenshots automatical...,Every time I take a screenshot and interact wi...,0,1
4150,I think someone has access to my icloud help me,,0,2
2585,Settings search still broke,On IOS 26.5 and IOS 27.3 Beta I still having p...,1,2
2499,WhatsApp instantly deletes restored data upon ...,,1,1
3742,Apple leasing is not a good deal,Twitter is buzzing with the leasing option and...,0,65
494,Why on my iphone 17 the back cover can be slip...,,0,0
511,Transferring my MP3's to new new iPhone.,I have tried to download both Apple Music for ...,3,10


**Lưu report**

In [ ]:
# Lưu tất cả các kết quả phân tích thống kê (profiling) ra định dạng file CSV tương ứng
overview_df = pd.DataFrame(
    [overview]
)

overview_df.to_csv(
    PROFILE_DIR / "posts_overview.csv",
    index=False
)

missing.to_csv(
    PROFILE_DIR / "posts_missing.csv",
    index=False
)

numeric_profile.to_csv(
    PROFILE_DIR / "posts_numeric.csv"
)

posts_per_day.to_csv(
    PROFILE_DIR / "posts_per_day.csv",
    index=False
)

## Tạo Bronze Parquet

In [ ]:
# Thiết lập biến môi trường chỉ đường dẫn dữ liệu Bronze đầu vào và đầu ra Parquet
POSTS_FILTERED = (
    FILTERED_DIR /
    "iphone_posts_2026_07.jsonl"
)

POSTS_PARQUET = (
    PARQUET_DIR /
    "iphone_posts_2026_07.parquet"
)

print("Input :", POSTS_FILTERED)
print("Output:", POSTS_PARQUET)

Input : /content/drive/MyDrive/DE/filtered/iphone_posts_2026_07.jsonl
Output: /content/drive/MyDrive/DE/parquet/iphone_posts_2026_07.parquet


In [ ]:
# Đọc lại file JSONL đã được lưu trữ làm nguồn trung gian để chuẩn bị chuyển đổi
bronze_posts = pd.read_json(
    POSTS_FILTERED,
    lines=True
)

print("Rows   :", len(bronze_posts))
print("Columns:", len(bronze_posts.columns))

Rows   : 4205
Columns: 121


In [ ]:
# Hiển thị tóm tắt cấu trúc trường của bộ dữ liệu chuẩn bị đưa vào kho lưu trữ Bronze
bronze_posts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4205 entries, 0 to 4204
Columns: 121 entries, _meta to author_cakeday
dtypes: bool(28), float64(28), int64(14), object(51)
memory usage: 3.1+ MB


In [ ]:
# Trích xuất danh sách tất cả các trường dữ liệu hiện có trong DataFrame
bronze_posts.columns.tolist()

['_meta',
 'all_awardings',
 'allow_live_comments',
 'approved_at_utc',
 'approved_by',
 'archived',
 'author',
 'author_flair_background_color',
 'author_flair_css_class',
 'author_flair_richtext',
 'author_flair_template_id',
 'author_flair_text',
 'author_flair_text_color',
 'author_flair_type',
 'author_fullname',
 'author_is_blocked',
 'author_patreon_flair',
 'author_premium',
 'awarders',
 'banned_at_utc',
 'banned_by',
 'can_gild',
 'can_mod_post',
 'category',
 'clicked',
 'content_categories',
 'contest_mode',
 'created',
 'created_utc',
 'crosspost_parent',
 'crosspost_parent_list',
 'discussion_type',
 'distinguished',
 'domain',
 'downs',
 'edited',
 'gilded',
 'gildings',
 'hidden',
 'hide_score',
 'id',
 'is_created_from_ads_ui',
 'is_crosspostable',
 'is_meta',
 'is_original_content',
 'is_reddit_media_domain',
 'is_robot_indexable',
 'is_self',
 'is_video',
 'likes',
 'link_flair_background_color',
 'link_flair_css_class',
 'link_flair_richtext',
 'link_flair_template_

In [ ]:
# Tìm và liệt kê các cột chứa định dạng phức tạp/lồng nhau (như dict, list)
nested_columns = []

for column in bronze_posts.columns:
    has_nested = bronze_posts[column].apply(
        lambda x: isinstance(x, (dict, list))
    ).any()
    if has_nested:
        nested_columns.append(column)

print("Nested columns:")
print(nested_columns)

Nested columns:
['_meta', 'all_awardings', 'author_flair_richtext', 'awarders', 'crosspost_parent_list', 'gildings', 'link_flair_richtext', 'media', 'media_embed', 'mod_reports', 'outbound_link', 'secure_media', 'secure_media_embed', 'treatment_tags', 'user_reports', 'preview', 'gallery_data', 'media_metadata']


In [ ]:
def serialize_nested(value):
    """
    Hàm tuần tự hóa cấu trúc lồng nhau (dict/list) sang chuỗi JSON string để ghi xuống định dạng Parquet.
    """
    if isinstance(value, (dict, list)):
        return json.dumps(
            value,
            ensure_ascii=False
        )
    return value

In [ ]:
# Áp dụng chuyển đổi định dạng chuỗi JSON cho tất cả các cột kiểu lồng phức tạp
for column in nested_columns:
    bronze_posts[column] = (
        bronze_posts[column]
        .apply(serialize_nested)
    )

In [ ]:
# Thêm các cột siêu dữ liệu (metadata) hỗ trợ truy vết lịch sử nguồn dữ liệu tích hợp
bronze_posts["_ingest_source"] = (
    "reddit_academic_torrents"
)

bronze_posts["_ingest_period"] = (
    "2026-07"
)

bronze_posts["_ingest_scope"] = (
    "r/iphone"
)

bronze_posts["_ingested_at_utc"] = (
    pd.Timestamp.now(tz="UTC")
)

In [ ]:
# Xác nhận cấu trúc và độ toàn vẹn của tập dữ liệu Bronze trước khi xuất file
print("===== BRONZE DATASET =====")
print("Rows:", len(bronze_posts))
print("Columns:", len(bronze_posts.columns))
print("Duplicate IDs:", bronze_posts["id"].duplicated().sum())
print("Subreddits:")
print(bronze_posts["subreddit"].value_counts())

===== BRONZE DATASET =====
Rows: 4205
Columns: 125
Duplicate IDs: 0
Subreddits:
subreddit
iphone    4205
Name: count, dtype: int64


In [ ]:
# Ghi DataFrame ra file Parquet chất lượng cao sử dụng thư viện PyArrow và giải thuật nén Snappy
bronze_posts.to_parquet(
    POSTS_PARQUET,
    engine="pyarrow",
    index=False,
    compression="snappy"
)

print("Bronze Parquet saved to:")
print(POSTS_PARQUET)

Bronze Parquet saved to:
/content/drive/MyDrive/DE/parquet/iphone_posts_2026_07.parquet


In [ ]:
# So sánh kích thước vật lý giữa tệp JSON ban đầu và tệp lưu trữ Parquet
json_size_mb = (
    POSTS_FILTERED.stat().st_size
    / 1024**2
)

parquet_size_mb = (
    POSTS_PARQUET.stat().st_size
    / 1024**2
)

print(f"Filtered JSONL : {json_size_mb:.2f} MB")
print(f"Bronze Parquet : {parquet_size_mb:.2f} MB")

Filtered JSONL : 18.00 MB
Bronze Parquet : 3.37 MB


In [ ]:
# Đọc thử tệp Parquet vừa lưu để bảo đảm dữ liệu có thể khôi phục chính xác
bronze_check = pd.read_parquet(
    POSTS_PARQUET
)

print("Rows:", len(bronze_check))
print("Columns:", len(bronze_check.columns))
bronze_check.head()

Rows: 4205
Columns: 125


,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,post_hint,preview,gallery_data,is_gallery,media_metadata,author_cakeday,_ingest_source,_ingest_period,_ingest_scope,_ingested_at_utc
0,"{""edited_title"": ""[ Removed by moderator ]"", ""...",[],False,NaN,NaN,False,Joseph930,None,None,[],...,None,None,None,NaN,None,NaN,reddit_academic_torrents,2026-07,r/iphone,2026-10-18 10:00:37.052518+00:00
1,"{""retrieved_2nd_on"": 1782993955}",[],False,NaN,NaN,False,Exotic-Ad-7209,None,None,[],...,None,None,None,NaN,None,NaN,reddit_academic_torrents,2026-07,r/iphone,2026-10-18 10:00:37.052518+00:00
2,"{""edited_title"": ""[ Removed by moderator ]"", ""...",[],False,NaN,NaN,False,AlmxstAlive,None,None,[],...,image,"{""enabled"": true, ""images"": [{""id"": ""b5vkt6jhf...",None,NaN,None,NaN,reddit_academic_torrents,2026-07,r/iphone,2026-10-18 10:00:37.052518+00:00
3,"{""retrieved_2nd_on"": 1782995656}",[],False,NaN,NaN,False,_straightasmyhair,None,None,[],...,None,None,None,NaN,None,NaN,reddit_academic_torrents,2026-07,r/iphone,2026-10-18 10:00:37.052518+00:00
4,"{""retrieved_2nd_on"": 1782995672}",[],False,NaN,NaN,False,frenetic_alien,None,None,[],...,None,None,None,NaN,None,NaN,reddit_academic_torrents,2026-07,r/iphone,2026-10-18 10:00:37.052518+00:00


In [ ]:
# Chạy các phép xác nhận logic nhằm ràng buộc toàn vẹn số dòng, trùng lặp và tính đúng đắn của dữ liệu
assert len(bronze_check) == 4205

assert (
    bronze_check["id"]
    .duplicated()
    .sum()
    == 0
)

assert (
    bronze_check["subreddit"]
    .str.lower()
    .eq("iphone")
    .all()
)

print("Bronze validation passed.")

Bronze validation passed.


In [ ]:
# Hiển thị một mẫu dữ liệu dạng cột-giá trị chi tiết để kiểm tra cấu trúc lưu trữ nội tại
sample = (
    bronze_check
    .iloc[0]
)

for key, value in sample.items():
    print(f"{key}: {value}")

_meta: {"edited_title": "[ Removed by moderator ]", "removal_type": "moderator", "retrieved_2nd_on": 1782993660, "was_deleted_later": true}
all_awardings: []
allow_live_comments: False
approved_at_utc: nan
approved_by: nan
archived: False
author: Joseph930
author_flair_background_color: None
author_flair_css_class: None
author_flair_richtext: []
author_flair_template_id: None
author_flair_text: None
author_flair_text_color: None
author_flair_type: text
author_fullname: t2_5aru5gfh
author_is_blocked: False
author_patreon_flair: 0.0
author_premium: 0.0
awarders: []
banned_at_utc: nan
banned_by: nan
can_gild: False
can_mod_post: False
category: nan
clicked: False
content_categories: nan
contest_mode: False
created: 1782864042
created_utc: 1782864042
crosspost_parent: t3_1uk5weu
crosspost_parent_list: [{"all_awardings": [], "allow_live_comments": false, "approved_at_utc": null, "approved_by": null, "archived": false, "author": "Joseph930", "author_flair_background_color": null, "author_fla

In [ ]:
# Thực hiện kiểm tra nhanh cuối cùng các thông số quan trọng của tệp lưu trữ Parquet
bronze = pd.read_parquet(POSTS_PARQUET)

print("Rows:", len(bronze))
print("Columns:", len(bronze.columns))
print("Duplicate IDs:", bronze["id"].duplicated().sum())
print(bronze["subreddit"].value_counts())

Rows: 4205
Columns: 125
Duplicate IDs: 0
subreddit
iphone    4205
Name: count, dtype: int64


In [ ]:
import hashlib

def sha256_file(path, chunk_size=1024 * 1024):
    """
    Tính mã băm SHA256 của tệp tin đích để hỗ trợ xác thực tính toàn vẹn và chống lỗi hỏng tệp.
    """
    sha = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            sha.update(chunk)
    return sha.hexdigest()

In [ ]:
# Tính toán và xuất mã băm SHA-256 cụ thể cho tệp Parquet vừa khởi tạo thành công
bronze_sha256 = sha256_file(POSTS_PARQUET)

print("SHA256:", bronze_sha256)

SHA256: 2e936feef628c1e9c5e4e33a00cd6627fdace9565aed737ce22ceb90ddd7ad32
